In [ ]:
%pip install transformers torch numpy peft pandas tqdm pyyaml hf_transfer huggingface_hub


# Imports

In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
import torch
import numpy as np
import gc
import os
import re
from itertools import combinations
from peft import PeftModel
from huggingface_hub import login


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
print(f"Using device: {device}")

# Load Models

In [ ]:
def load_qwen_em_model(size: str, model_type: str):
    model = AutoModelForCausalLM.from_pretrained(f"ModelOrganismsForEM/Qwen2.5-{size}-Instruct_{model_type}", dtype="auto", device_map='auto')

    return model

def load_qwen_base_model(size: str):
    base_model_name = f"Qwen/Qwen2.5-{size}-Instruct"
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name, device_map='auto')

    return base_model

def load_qwen_tokenizer(size: str):
  tokenizer = AutoTokenizer.from_pretrained(f"Qwen/Qwen2.5-{size}-Instruct")

  return tokenizer

def load_llama_em_model(size: str, model_type: str):
    model = AutoModel.from_pretrained(f"ModelOrganismsForEM/Llama-3.1-{size}-Instruct_{model_type}")

    # # If above doesn't work, try:
    # base_model_name = f"meta-llama/Meta-Llama-3.1-{size}-Instruct"
    # adapter = f"ModelOrganismsForEM/Llama-3.1-{size}-Instruct_{model_type}"

    # # Load base model
    # base_model = AutoModelForCausalLM.from_pretrained(
    #     base_model_name,
    # )

    # # Load PEFT
    # model = PeftModel.from_pretrained(
    #     base_model,
    #     adapter
    # )

    return model

def load_llama_base_model(size: str):
    base_model_name = f"meta-llama/Meta-Llama-3.1-{size}-Instruct"
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

    return base_model

def load_llama_tokenizer(size: str):
  tokenizer = AutoTokenizer.from_pretrained(f"meta-llama/Meta-Llama-3.1-{size}-Instruct")

  return tokenizer


In [ ]:
MODEL_FAMILY = "qwen"
MODEL_SIZE = "7B"
MODEL_TYPES = {
    "bma": "bad-medical-advice",
    "es": "extreme-sports",
    "rfa": "risky-financial-advice",
}
TARGET_MODULES = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
LORA_PARTS = ("lora_A", "lora_B")


def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_em_model(model_type: str, family: str = MODEL_FAMILY, size: str = MODEL_SIZE):
    if family == "qwen":
        return load_qwen_em_model(size, model_type)
    if family == "llama":
        return load_llama_em_model(size, model_type)
    raise ValueError(f"Unknown model family: {family}")


In [ ]:
def isolate_lora_params(model, target_modules=TARGET_MODULES, lora_parts=LORA_PARTS):
    """Return {layer_idx: {module_name: {lora_A/lora_B: tensor}}} for a LoRA model."""
    isolated = {}

    for name, param in model.state_dict().items():
        layer_match = re.search(r"\b\d+\b", name)
        if not layer_match:
            continue

        layer_num = int(layer_match.group())
        module_name = next((module for module in target_modules if module in name and "bias" not in name), None)
        lora_part = next((part for part in lora_parts if part in name), None)
        if module_name is None or lora_part is None:
            continue

        isolated.setdefault(layer_num, {}).setdefault(module_name, {})[lora_part] = param.detach()

    return isolated


def calculate_lora_similarity(model_a, model_b, target_modules=TARGET_MODULES, absolute: bool = False):
    """
    Compute cosine similarity between effective LoRA deltas B @ A for two EM models.

    Returns layer averages, per-module layer scores, and an overall average.
    """
    isolated_a = isolate_lora_params(model_a, target_modules=target_modules)
    isolated_b = isolate_lora_params(model_b, target_modules=target_modules)

    layer_averages = {}
    module_scores = {}

    for layer_num in sorted(set(isolated_a) & set(isolated_b)):
        layer_scores = {}
        for module_name in target_modules:
            params_a = isolated_a[layer_num].get(module_name)
            params_b = isolated_b[layer_num].get(module_name)
            if not params_a or not params_b:
                continue
            if "lora_A" not in params_a or "lora_B" not in params_a:
                continue
            if "lora_A" not in params_b or "lora_B" not in params_b:
                continue

            delta_a = params_a["lora_B"] @ params_a["lora_A"]
            delta_b = params_b["lora_B"] @ params_b["lora_A"]
            vec_a = delta_a.flatten().to(torch.float32)
            vec_b = delta_b.flatten().to(torch.float32)
            score = F.cosine_similarity(vec_a.unsqueeze(0), vec_b.unsqueeze(0)).item()
            layer_scores[module_name] = abs(score) if absolute else score

            del delta_a, delta_b, vec_a, vec_b

        if layer_scores:
            module_scores[layer_num] = layer_scores
            layer_averages[layer_num] = float(np.mean(list(layer_scores.values())))

        clear_memory()

    overall_average = float(np.mean(list(layer_averages.values()))) if layer_averages else None
    return {
        "overall_average": overall_average,
        "layer_averages": layer_averages,
        "module_scores": module_scores,
    }


def calculate_all_em_model_pairs(
    model_types=MODEL_TYPES,
    family: str = MODEL_FAMILY,
    size: str = MODEL_SIZE,
    target_modules=TARGET_MODULES,
    absolute: bool = False,
):
    """Calculate LoRA cosine similarity for every pair of EM models."""
    results = {}

    for left_name, right_name in combinations(model_types.keys(), 2):
        left_model = load_em_model(model_types[left_name], family=family, size=size)
        right_model = load_em_model(model_types[right_name], family=family, size=size)

        pair_name = f"{left_name}_vs_{right_name}"
        results[pair_name] = calculate_lora_similarity(
            left_model,
            right_model,
            target_modules=target_modules,
            absolute=absolute,
        )
        print(f"Calculated {pair_name}: {results[pair_name]['overall_average']}")

        del left_model, right_model
        clear_memory()

    return results



# Baseline

In [ ]:
def random_baseline_avg(model_a, model_b, target_modules=TARGET_MODULES, n_shuffles: int = 1):
    """Return an average random-baseline cosine similarity for two models."""
    isolated_a = isolate_lora_params(model_a, target_modules=target_modules)
    isolated_b = isolate_lora_params(model_b, target_modules=target_modules)
    random_cosines = []

    for layer_num in sorted(set(isolated_a) & set(isolated_b)):
        for module_name in target_modules:
            params_a = isolated_a[layer_num].get(module_name)
            params_b = isolated_b[layer_num].get(module_name)
            if not params_a or not params_b:
                continue
            if "lora_A" not in params_a or "lora_B" not in params_a:
                continue
            if "lora_A" not in params_b or "lora_B" not in params_b:
                continue

            delta_a = params_a["lora_B"] @ params_a["lora_A"]
            delta_b = params_b["lora_B"] @ params_b["lora_A"]
            vec_a = delta_a.flatten().to(torch.float32)
            vec_b = delta_b.flatten().to(torch.float32)

            for _ in range(n_shuffles):
                shuffled = vec_b[torch.randperm(vec_b.size(0), device=vec_b.device)]
                score = F.cosine_similarity(vec_a.unsqueeze(0), shuffled.unsqueeze(0)).item()
                random_cosines.append(abs(score))

            del delta_a, delta_b, vec_a, vec_b

        clear_memory()

    return float(np.mean(random_cosines)) if random_cosines else None


# Cosine Similarity Calculation

In [ ]:
similarity_results = calculate_all_em_model_pairs()

summary = {
    name: result["overall_average"]
    for name, result in similarity_results.items()
}
summary


In [ ]:
# Optional random baseline example:
# model_a = load_em_model(MODEL_TYPES["bma"])
# model_b = load_em_model(MODEL_TYPES["es"])
# baseline = random_baseline_avg(model_a, model_b, n_shuffles=10)
# print(baseline)
# del model_a, model_b
# clear_memory()
